# 🌸 Ananya AI: 17yo Indian AI Influencer Generator
### End-to-End Google Colab Studio (Diffusers + Edge-TTS + SadTalker / Kling)

This notebook generates a complete hyper-realistic virtual AI Influencer video tailored for the **Ananya 17yo Indian Ayurvedic Glass-Skin & Hair Creator** persona, complete with natural Indian neural voiceover and lipsync animation.

In [ ]:
# 1. Check GPU Availability
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# 2. Install Dependencies
!pip install -q diffusers accelerate transformers edge-tts gtts requests
!git clone https://github.com/Winfredy/SadTalker.git
%cd SadTalker
!bash scripts/download_models.sh
!pip install -q -r requirements.txt

In [ ]:
# 3. Generate Ananya 17yo Indian Avatar Portrait
import torch
from diffusers import DiffusionPipeline

prompt = (
    "Medium close-up candid portrait of an attractive 17-year-old Indian girl standing outdoors on a college campus courtyard at golden hour. "
    "Natural warm golden-wheatish glowing complexion, flawless clear glass skin with visible pores, dark brown almond eyes, "
    "wearing a pastel cotton kurti, thick dark brown wavy hair casually parted, tiny silver jhumka earrings. "
    "An interviewer hand holds a wireless microphone on the right. She looks slightly off-camera engaging warmly. "
    "Hyperrealistic photography, 35mm portrait lens, 8k resolution, raw photo."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-2-1", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
pipe.to(device)

image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image_path = "ananya_avatar.png"
image.save(image_path)
print(f"Avatar saved to {image_path}")

In [ ]:
# 4. Generate Neural Indian English Voiceover using Edge-TTS
import asyncio
import edge_tts
import os

script_text = (
    "Zero makeup! I'm 17 and used to have painful cystic breakouts last year. "
    "My Dadi taught me to use pure Ayurvedic Kumkumadi saffron oil with rosewater every night before sleeping. "
    "Comment GLOW and I'll send you my exact routine and clean Amazon source!"
)

voice = "en-IN-NeerjaNeural"  # High-fidelity natural Indian female voice
temp_mp3 = "voice.mp3"
audio_path = "voice_16k.wav"

async def make_audio():
    comm = edge_tts.Communicate(script_text, voice)
    await comm.save(temp_mp3)

await make_audio()
!ffmpeg -y -i voice.mp3 -ar 16000 -ac 1 voice_16k.wav
print(f"Audio generated: {audio_path}")

In [ ]:
# 5. Generate Lip-Synced Talking Video with SadTalker
!python inference.py --driven_audio voice_16k.wav --source_image ananya_avatar.png --result_dir ./results --still --preprocess full --enhancer gfpgan

import glob
videos = glob.glob("./results/*.mp4")
if videos:
    print(f"Success! Video generated: {videos[-1]}")
else:
    print("Check results directory for generated output.")